In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import lightgbm as lgb
import joblib

# ========================
# 1. Load Data
# ========================
df = pd.read_csv("training_data.csv")
print(f"Total readings: {len(df)}")
print(f"Anomalies: {df['is_anomaly'].sum()} ({df['is_anomaly'].mean()*100:.1f}%)")

# ========================
# 2. Features & Target
# ========================
FEATURES = ["power_consumption", "voltage", "current",
            "temperature", "power_factor", "frequency"]

X = df[FEATURES]
y = df["is_anomaly"]

# ========================
# 3. Split
# ========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ========================
# 4. Scale
# ========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ========================
# 5. Train LightGBM
# ========================
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    is_unbalance=True,   # بيتعامل مع الـ 95/5 تلقائياً
    random_state=42,
    verbose=-1
)

model.fit(X_train_scaled, y_train)

# ========================
# 6. Evaluate
# ========================
y_pred = model.predict(X_test_scaled)

print("\n--- Results ---")
print(classification_report(y_test, y_pred, target_names=["Normal", "Anomaly"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ========================
# 7. Save
# ========================
joblib.dump(model,  "lgbm_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("\n lgbm_model.pkl")
